In [2]:
from pyspark.sql.functions import col, avg, count, when,  sum as spark_sum, round, max, min

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 4, Finished, Available, Finished, False)

In [3]:
# Read Silver table

df_silver = spark.read\
    .format("delta")\
    .table("silver_patient_records")

print(f"silver rows loaded: {df_silver.count()}")
display(df_silver.limit(5))

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 5, Finished, Available, Finished, False)

silver rows loaded: 55500


SynapseWidget(Synapse.DataFrame, 6bf31a49-9f7c-4242-b241-3393e9416baf)

In [4]:
# Gold Table 1: Medical Condition Summary


gold_medical = df_silver.groupBy("Medical_Condition").agg(
    count("*").alias("Total_Patients"),
    round(avg("Billing_Amount"), 2).alias("Avg_Billing"),
    round(avg("Length_of_Stay"), 1).alias("Avg_Stay_Days"),
    spark_sum(when(col("Risk_Flag") == "High Risk", 1).otherwise(0)).alias("High_Risk_Patients"),
    spark_sum(when(col("Test_Results") == "ABNORMAL", 1).otherwise(0)).alias("Abnormal_Results"),
    spark_sum(when(col("High_Billing_Flag") == True, 1).otherwise(0)).alias("High_Billing_Cases"),
    round(avg( when( col("Risk_Flag") == "High Risk", col("Billing_Amount"))), 2 ).alias("Avg_HighRisk_Billing")
).orderBy("Total_Patients", ascending=False)

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 6, Finished, Available, Finished, False)

In [5]:
display(gold_medical)

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 48cc35f4-28ce-4f75-967d-72a9aac5c002)

In [6]:
# Gold Table 2: Hospital Performance Metrics

gold_hospital = df_silver.groupBy("Hospital").agg(
    count("*").alias("Total_Patients"),
    round(avg("Billing_Amount"), 2).alias("Avg_Billing"),
    round(avg("Length_of_Stay"), 1).alias("Avg_Stay_Days"),
    spark_sum(when(col("Risk_Flag") == "High Risk", 1).otherwise(0)).alias("High_Risk_Cases"),
    spark_sum(when(col("Admission_Type") == "EMERGENCY", 1).otherwise(0)).alias("Emergency_Cases"),
    spark_sum(when(col("Admission_Type") == "URGENT", 1).otherwise(0)).alias("Urgent_Cases"),
    spark_sum(when(col("High_Billing_Flag") == True, 1).otherwise(0)).alias("High_Billing_Cases"),
    round(spark_sum("Billing_Amount"), 2).alias("Total_Revenue")
).orderBy("Total_Patients", ascending=False)

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 8, Finished, Available, Finished, False)

In [7]:
display(gold_hospital)

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a06286ba-b83d-44c5-a798-795d3930f6bb)

In [8]:
# Gold Table 3: Doctor Performance


gold_doctor = df_silver.groupBy("Doctor", "Hospital").agg(
    count("*").alias("Total_Patients"),
    round(avg("Billing_Amount"), 2).alias("Avg_Billing"),
    round(avg("Length_of_Stay"), 1).alias("Avg_Stay_Days"),
    spark_sum(when(col("Risk_Flag") == "High Risk", 1).otherwise(0)).alias("High_Risk_Patients"),
    spark_sum(when(col("Test_Results") == "ABNORMAL", 1).otherwise(0)).alias("Abnormal_Cases"),
    spark_sum(when(col("Test_Results") == "NORMAL", 1).otherwise(0)).alias("Normal_Cases"),
    round(spark_sum("Billing_Amount"), 2).alias("Total_Revenue_Generated")
).orderBy("Total_Patients", ascending=False)

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 10, Finished, Available, Finished, False)

In [9]:
display(gold_doctor)

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f91bae97-13cb-432c-825e-e295dc1230c8)

In [16]:
# Gold Table 4: Patient Demographics & Risk Summary

gold_demographics = df_silver.groupBy("Age_Group", "Billing_Category", "Stay_Category").agg(
    count("*").alias("Total_Patients"),
    round(avg("Billing_Amount"), 2).alias("Avg_Billing"),
    round(avg("Length_of_Stay"), 1).alias("Avg_Stay_Days"),
    spark_sum(when(col("Risk_Flag") == "High Risk", 1).otherwise(0)).alias("High_Risk_Patients"),
    spark_sum(when(col("Risk_Flag") == "Low Risk", 1).otherwise(0)).alias("Low_Risk_Patients"),
    spark_sum(when(col("High_Billing_Flag") == True, 1).otherwise(0)).alias("High_Billing_Cases")
).orderBy("Age_Group", "Billing_Category")

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 18, Finished, Available, Finished, False)

In [17]:
display(gold_demographics)

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 90a4a6e7-bf4a-4b7f-85f2-b811a4f4e920)

In [21]:
# Gold Table 5: Insurance Provider Analysis

gold_insurance = df_silver.groupBy("Insurance_Provider").agg(
    count("*").alias("Total_Patients"),
    round(avg("Billing_Amount"), 2).alias("Avg_Billing"),
    round(spark_sum("Billing_Amount"), 2).alias("Total_Billed"),
    spark_sum(when(col("High_Billing_Flag") == True, 1).otherwise(0)).alias("High_Billing_Cases"),
    spark_sum(when(col("Risk_Flag") == "High Risk", 1).otherwise(0)).alias("High_Risk_Patients"),
    round(avg("Length_of_Stay"), 1).alias("Avg_Stay_Days")
).orderBy("Total_Billed", ascending=False)

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 23, Finished, Available, Finished, False)

In [22]:
display(gold_insurance)

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b4d903f6-62f9-4deb-bb2b-742aec15d6f5)

# Save all Gold tables


In [13]:
gold_medical.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("gold_medical_condition_summary")    

print("gold_medical_condition_summary saved!")


StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 15, Finished, Available, Finished, False)

gold_medical_condition_summary saved!


In [14]:
gold_hospital.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("gold_hospital_metrics")    

print("gold_hospital_metrics saved!")


StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 16, Finished, Available, Finished, False)

gold_hospital_metrics saved!


In [15]:
gold_doctor.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("gold_doctor_performance") 

print("gold_doctor_performance saved!")   

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 17, Finished, Available, Finished, False)

gold_doctor_performance saved!


In [18]:
gold_demographics.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("gold_demographics_summary") 

print("gold_demographics_summary saved!")   

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 20, Finished, Available, Finished, False)

gold_demographics_summary saved!


In [23]:
gold_insurance.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("gold_insurance_analysis") 

print("gold_insurance_analysis saved!")   

StatementMeta(, 6b0fd88d-6711-4243-9022-46a819f0cb34, 25, Finished, Available, Finished, False)

gold_insurance_analysis saved!
